[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Type Affinity &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the stations and their year of readings, in the two
tables the notebook used, and opens a connection that the tasks share. Run it first. The tasks do not
depend on one another, and the last cell closes the connection and removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
import zlib
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


conn = sqlite3.connect(DATABASE)
conn.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
conn.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                 ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
conn.commit()

print("built", DATABASE)


built scratch/stations.db


**1.** A column declared `BIGINT`.


In [2]:
conn.execute("CREATE TABLE counts (value BIGINT)")
conn.executemany("INSERT INTO counts VALUES (?)", [("7",), ("seven",)])

print(conn.execute("SELECT value, typeof(value) FROM counts").fetchall())


[(7, 'integer'), ('seven', 'text')]


`BIGINT` contains `INT`, so the column has INTEGER affinity. `'7'` became the integer 7, and
`'seven'`, which is no number, stayed text in a column declared as a big integer.


**2.** Four declared types, worked out and checked.


In [3]:
for declared in ["CHARACTER(10)", "DOUBLE PRECISION", "POINT", "BOOLEAN"]:
    conn.execute("DROP TABLE IF EXISTS guess")
    conn.execute(f"CREATE TABLE guess (value {declared})")
    conn.execute("INSERT INTO guess VALUES ('1')")
    print(f"{declared:<16}", conn.execute("SELECT value, typeof(value) FROM guess").fetchone())


CHARACTER(10)    ('1', 'text')
DOUBLE PRECISION (1.0, 'real')
POINT            (1, 'integer')
BOOLEAN          (1, 'integer')


`CHARACTER(10)` contains `CHAR`, so it is TEXT and kept `'1'` as text. `DOUBLE PRECISION` contains
`DOUB`, so it is REAL and stored 1.0. `POINT` contains `INT`, so it is INTEGER, and `BOOLEAN` matches
no rule, so it is NUMERIC. Both stored the integer 1, since INTEGER and NUMERIC store values the same
way.


**3.** `TEXT` and `ANY` in a `STRICT` table.


In [4]:
conn.execute("CREATE TABLE labels (label TEXT, anything ANY) STRICT")
conn.executemany("INSERT INTO labels VALUES (?, ?)", [(42, 42), (4.5, 4.5)])

for row in conn.execute("SELECT label, typeof(label), anything, typeof(anything) FROM labels"):
    print(row)


('42', 'text', 42, 'integer')
('4.5', 'text', 4.5, 'real')


A STRICT `TEXT` column still converts what converts without loss, so both numbers became text. `ANY`
kept each value exactly as it arrived, an integer and a real.


**4.** A `CHECK` on `typeof`.


In [5]:
conn.execute("CREATE TABLE guarded (celsius REAL CHECK (typeof(celsius) IN ('real', 'null')))")

for celsius in [-3.5, "4.5", "n/a"]:
    try:
        conn.execute("INSERT INTO guarded VALUES (?)", (celsius,))
        print(f"{celsius!r:<6} accepted")
    except sqlite3.IntegrityError as error:
        print(f"{celsius!r:<6} refused: {error}")
print(conn.execute("SELECT celsius, typeof(celsius) FROM guarded").fetchall())


-3.5   accepted
'4.5'  accepted
'n/a'  refused: CHECK constraint failed: typeof(celsius) IN ('real', 'null')
[(-3.5, 'real'), (4.5, 'real')]


The check runs after the column's REAL affinity has converted the value, so `'4.5'` arrived at it as
the number 4.5 and passed. `'n/a'` stayed text and failed it.


**5.** A day of Oslo's readings as a blob.


In [6]:
day = conn.execute("""
    SELECT r.hour, r.celsius FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = ? AND r.hour LIKE ? ORDER BY r.hour
""", ("Oslo", "2025-03-01%")).fetchall()
text = "\n".join(f"{hour},{celsius}" for hour, celsius in day).encode()

conn.execute("CREATE TABLE day_archives (id INTEGER PRIMARY KEY, data BLOB NOT NULL) STRICT")
archive_id = conn.execute("INSERT INTO day_archives (data) VALUES (?)", (zlib.compress(text),)).lastrowid
conn.commit()

with conn.blobopen("day_archives", "data", archive_id, readonly=True) as blob:
    print("first two bytes:", blob.read(2))
(data,) = conn.execute("SELECT data FROM day_archives WHERE id = ?", (archive_id,)).fetchone()
print("first line:", zlib.decompress(data).decode().splitlines()[0])


first two bytes: b'x\x9c'
first line: 2025-03-01T00:00,-1.7


`blobopen` read two bytes of the stored value without fetching the rest, and `SELECT` fetched all of
it, as `bytes`, for `zlib.decompress` to turn back into the text that went in.


**6.** Which tables are `STRICT`.


In [7]:
for name, strict in conn.execute("""
    SELECT name, strict FROM pragma_table_list
    WHERE schema = 'main' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
"""):
    print(f"{name:<13} {'STRICT' if strict else 'flexible'}")


counts        flexible
day_archives  STRICT
guarded       flexible
guess         flexible
labels        STRICT
readings      flexible
stations      flexible


`pragma_table_list` lists every table with a `strict` column that is 1 or 0. `schema = 'main'` keeps
to the database file itself, and `NOT LIKE 'sqlite_%'` drops SQLite's own tables, such as
`sqlite_schema`.

Last, close the connection and remove the scratch folder:


In [8]:
conn.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Type Affinity](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/08-type-affinity.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
